In [50]:
!pip install -q PySastrawi emoji

import re, html, time, joblib
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import emoji
import nltk
from tqdm.auto import tqdm
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords', quiet=True)
tqdm.pandas()
pd.set_option('display.max_colwidth', 120)

RANDOM_STATE = 42   # agar hasil split selalu sama (reproducible)
print("Library siap.")

Library siap.


In [51]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [52]:
import pandas as pd

# Menggunakan file CSV yang terdeteksi di Drive kamu
DATA_PATH = "/content/drive/MyDrive/data_labeling.csv"

try:
    df = pd.read_csv(DATA_PATH)
    print("Berhasil memuat data!")
    print(f"Ukuran data: {df.shape}")
    display(df.head())
except Exception as e:
    print(f"Gagal membaca file: {e}")

Berhasil memuat data!
Ukuran data: (13637, 13)


,video_id,video_title,comment_id,is_reply,parent_id,author,text,likeCount,publishedAt,is_relevant_keyword,emoji_count,emoji_str,label_annotator
0,sfeRpL4t6yY,Menteri Risma Dikritik Masyarakat Usai Paksa Penyandang Tunarungu Lakukan Hal Ini | Kabar Petang,Ugy__Xbktb9UbUgv5M94AaABAg,False,NaN,@ourthings5977,dongo,0.0,2023-12-03T16:21:48Z,False,0.0,NaN,eksklusi
1,sfeRpL4t6yY,Menteri Risma Dikritik Masyarakat Usai Paksa Penyandang Tunarungu Lakukan Hal Ini | Kabar Petang,Ugxmuu_b9zZ0DU_Ptad4AaABAg,False,NaN,@Hellebæk-Ålsgårde_009,Geram gue liat ibu mentri menolak kodrat Allah.,0.0,2023-08-25T01:00:25Z,False,0.0,NaN,eksklusi
2,sfeRpL4t6yY,Menteri Risma Dikritik Masyarakat Usai Paksa Penyandang Tunarungu Lakukan Hal Ini | Kabar Petang,UgwGj1x5QwlQQHqiYO54AaABAg,False,NaN,@elmiutami1515,"Maklumi aja,,namanya juga bukan guru disabilitas ,,gk usah dipanjang2kan dn di plintir2kan ntar timbul masalah baru...",0.0,2023-02-16T05:41:07Z,True,0.0,NaN,eksklusi
3,HqYHDaPKWn8,Supir tunarungu mendapat penumpang spesial,UgycSjbhAcoghDks6Uh4AaABAg,False,NaN,@indahnganjuk9302,Teharu🙏 penupang,0.0,2025-10-08T11:28:36Z,False,1.0,🙏,inklusi
4,HqYHDaPKWn8,Supir tunarungu mendapat penumpang spesial,Ugy9MRkhly_nWufIUAx4AaABAg,False,NaN,@dsrcourse9407,"Bahasa terindah yg ingin ku pelajari, ya bahasa isyarat ❤",0.0,2025-10-06T11:08:06Z,True,1.0,❤,inklusi


### Eksplorasi data awal & pembersihan rule-based

In [53]:
LABEL_COL = 'label_annotator'

print("Jumlah baris  :", len(df))
print("Jumlah kolom  :", df.shape[1], "->", list(df.columns))
print("\nMissing value per kolom:")
print(df.isna().sum()[lambda s: s > 0])
print("\nDistribusi label (data mentah):")
print(df[LABEL_COL].value_counts(dropna=False))

Jumlah baris  : 13637
Jumlah kolom  : 13 -> ['video_id', 'video_title', 'comment_id', 'is_reply', 'parent_id', 'author', 'text', 'likeCount', 'publishedAt', 'is_relevant_keyword', 'emoji_count', 'emoji_str', 'label_annotator']

Missing value per kolom:
video_id                  24
video_title               24
comment_id                24
is_reply                  24
parent_id              12575
author                    24
text                      24
likeCount                 24
publishedAt               24
is_relevant_keyword       24
emoji_count               24
emoji_str               8820
dtype: int64

Distribusi label (data mentah):
label_annotator
inklusi     9571
eksklusi    4066
Name: count, dtype: int64


In [54]:
df_cleaned = df.copy()
funnel = [("Data mentah", len(df_cleaned))]

# Normalisasi minimal teks: unescape HTML + trim spasi
df_cleaned['text'] = df_cleaned['text'].astype('string').map(lambda t: html.unescape(t).strip() if pd.notna(t) else t)

# (1) Nilai kosong
df_cleaned = df_cleaned[df_cleaned['text'].notna() & (df_cleaned['text'] != '')]
funnel.append(("Setelah hapus missing value", len(df_cleaned)))

# (2) Duplikat persis
df_cleaned = df_cleaned.drop_duplicates(subset='text', keep='first')
funnel.append(("Setelah hapus duplikat", len(df_cleaned)))

# (3) Komentar < 3 karakter
df_cleaned = df_cleaned[df_cleaned['text'].str.len() >= 3]
funnel.append(("Setelah hapus komentar < 3 karakter", len(df_cleaned)))

# (4) Komentar tanpa huruf (hanya emoji / simbol / angka)
df_cleaned = df_cleaned[df_cleaned['text'].str.contains(r'[A-Za-z]', regex=True)]
funnel.append(("Setelah hapus komentar tanpa huruf", len(df_cleaned)))

df_cleaned = df_cleaned.reset_index(drop=True)
df_cleaned['text'] = df_cleaned['text'].astype(str)

funnel_df = pd.DataFrame(funnel, columns=['Tahap', 'Jumlah baris'])
funnel_df['Dihapus'] = (-funnel_df['Jumlah baris'].diff()).fillna(0).astype(int)
funnel_df

,Tahap,Jumlah baris,Dihapus
0,Data mentah,13637,0
1,Setelah hapus missing value,13613,24
2,Setelah hapus duplikat,13613,0
3,Setelah hapus komentar < 3 karakter,13613,0
4,Setelah hapus komentar tanpa huruf,13613,0


### Preprocessing teks

In [55]:
import os
import Sastrawi

# ---------- Kamus normalisasi (slang -> baku) ----------
URL_KAMUS = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
kamus = pd.read_csv(URL_KAMUS).drop_duplicates(subset='slang', keep='first')
NORM_DASAR = dict(zip(kamus['slang'], kamus['formal']))

# Kamus kata dasar bawaan Sastrawi: dipakai untuk memperbaiki huruf memanjang & mengukur kualitas
ROOT_WORDS = set(open(os.path.join(os.path.dirname(Sastrawi.__file__), 'Stemmer', 'data', 'kata-dasar.txt')).read().split())

# Aturan dari versi awal Anda
NORM_CUSTOM_AWAL = {
    'gak': 'tidak', 'ga': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak', 'enggak': 'tidak',
    'engga': 'tidak', 'gk': 'tidak', 'tdk': 'tidak', 'tak': 'tidak', 'kaga': 'tidak', 'kagak': 'tidak',
    'yg': 'yang', 'krn': 'karena', 'karna': 'karena', 'dgn': 'dengan', 'utk': 'untuk',
    'kalo': 'kalau', 'klo': 'kalau', 'tau': 'tahu', 'gue': 'saya', 'gw': 'saya', 'bgt': 'banget',
    'tuli': 'tuli', 'bisu': 'bisu', 'tunarungu': 'tunarungu', 'disabilitas': 'disabilitas',
    'difabel': 'difabel', 'kerja': 'kerja',
}
# Tambahan pada versi perbaikan
NORM_CUSTOM_BARU = {
    # negasi (termasuk bahasa daerah) -> 'tidak'
    'g': 'tidak', 'teu': 'tidak', 'henteu': 'tidak', 'ora': 'tidak', 'ndak': 'tidak', 'nda': 'tidak',
    'ndk': 'tidak', 'gaak': 'tidak', 'gpp': 'tidak apa', 'gabisa': 'tidak bisa', 'gaboleh': 'tidak boleh',
    'bukannya': 'bukan', 'bukankah': 'bukan', 'bukanlah': 'bukan', 'belumlah': 'belum',
    'janganlah': 'jangan', 'tidaklah': 'tidak', 'tidakkah': 'tidak',
    # ucapan terima kasih -> satu bentuk
    'terimakasih': 'terima kasih', 'trimakasih': 'terima kasih', 'makasih': 'terima kasih',
    'makasi': 'terima kasih', 'mksh': 'terima kasih', 'trims': 'terima kasih', 'nuhun': 'terima kasih',
    'terimaksih': 'terima kasih', 'haturnuhun': 'terima kasih', 'thanks': 'terima kasih', 'thank': 'terima kasih',
    # ungkapan religius -> satu bentuk
    'masyaallah': 'masyaallah', 'masyallah': 'masyaallah', 'masyaalloh': 'masyaallah',
    'masyaaallah': 'masyaallah', 'alhamdulilah': 'alhamdulillah', 'amiin': 'amin', 'aamiin': 'amin',
    # singkatan / slang lain
    'bpk': 'bapak', 'bp': 'bapak', 'gub': 'gubernur', 'gubenur': 'gubernur', 'mantab': 'mantap',
    'muantap': 'mantap', 'rame': 'ramai', 'vidio': 'video', 'difable': 'difabel', 'ijin': 'izin',
    'sodara': 'saudara', 'komen': 'komentar', 'lu': 'kamu', 'lo': 'kamu', 'aing': 'saya', 'urang': 'saya',
    'abdi': 'saya', 'pisan': 'sekali', 'ku': 'aku', 'si': 'si', 'd': 'di', 'ortu': 'orang tua',
}
# Koreksi hasil kamus yang keliru / kata yang belum ternormalisasi (REVISI)
NORM_KOREKSI = {
    'pa': 'bapak',      # kamus lama: 'pa' -> 'apa' (salah, konteks "PA KDM")
    'rakyt': 'rakyat', 'klu': 'kalau', 'kya': 'kayak', 'syg': 'sayang',
}
NORM_AWAL = {**NORM_DASAR, **NORM_CUSTOM_AWAL}
NORM = {**NORM_AWAL, **NORM_CUSTOM_BARU, **NORM_KOREKSI}

# ---------- Stopword ----------
NEGASI = {'tidak', 'tak', 'bukan', 'belum', 'jangan', 'tanpa', 'kurang',
          'nggak', 'gak', 'ga', 'enggak', 'engga', 'tdk', 'gk'}
STOP_TAMBAHAN_AWAL = {'nya', 'ya', 'sih', 'nih', 'tuh', 'deh', 'dong', 'kok', 'lah', 'kan', 'mah', 'yah',
                      'kak', 'bang', 'mbak', 'kang', 'the', 'wkwk', 'wkwkwk', 'haha', 'hehe'}
STOP_TAMBAHAN_BARU = STOP_TAMBAHAN_AWAL | {'si', 'bapak', 'pak', 'bung', 'bro', 'aku', 'kamu',
                      'an', 'ber', 'ah', 'eh', 'oh', 'euy', 'atuh', 'lho', 'wae', 'nu', 'ti', 'aya',
                      'sareng', 'hayu', 'wr', 'wb'}

# Kata dalam daftar NLTK yang SENGAJA dipertahaman karena membawa makna untuk inklusi/eksklusi
KEEP_WORDS = {
    # modalitas / kemampuan  ("tidak BISA", "tidak BOLEH", "MAMPU bekerja")
    'bisa', 'bisakah', 'boleh', 'bolehkah', 'bolehlah', 'dapat', 'mampu', 'mampukah', 'semampu', 'harus',
    'haruslah', 'harusnya', 'seharusnya', 'sebaiknya', 'sepantasnya', 'pantas', 'perlu', 'perlukah', 'usah',
    'percuma', 'mau', 'ingin', 'inginkah', 'hendak',
    # evaluasi  ("LUAR BIASA", "BAIK", "BENAR")
    'baik', 'sebaik', 'benar', 'betul', 'benarkah', 'biasa', 'luar', 'penting', 'pentingnya',
    'keterlaluan', 'terlalu', 'cukup', 'jelas',
    # pekerjaan (topik utama penelitian)
    'bekerja', 'mengerjakan', 'dikerjakan',
    # kata tanya / keraguan
    'apa', 'apakah', 'apaan', 'kapan', 'kenapa', 'mengapa', 'bagaimana', 'bagaimanakah', 'mana',
    'siapa', 'berapa', 'mungkin', 'masih',
    # pronomina kelompok (kita vs mereka)
    'mereka', 'kita', 'kami', 'kalian',
    # dukungan singkat (REVISI: sebelumnya 'siap' dan 'lanjut' terbuang sebagai stopword)
    'siap', 'lanjut',
    # konjungsi kontras
    'tapi', 'tetapi', 'namun', 'padahal', 'walau', 'walaupun', 'meski', 'meskipun', 'hanya', 'cuma', 'malah', 'justru',
}
NLTK_STOP = set(stopwords.words('indonesian'))
STOP_AWAL = (NLTK_STOP | STOP_TAMBAHAN_AWAL) - NEGASI
STOP_BARU = (NLTK_STOP | STOP_TAMBAHAN_BARU) - NEGASI - KEEP_WORDS

# Opsional: buang nama diri yang spesifik video (kdm, hasni, matahari, dst.). Default MATI,
# karena pada evaluasi di bawah menghapusnya justru menurunkan skor.
HAPUS_ENTITAS_VIDEO = False
ENTITAS_VIDEO = {'kdm', 'dedi', 'dedy', 'mulyadi', 'hasni', 'hasna', 'zidan', 'steven', 'adel', 'sanjaya',
                 'ivan', 'udin', 'koko', 'matahari', 'store', 'dept', 'dep', 'department', 'depstore'}

# ---------- Stemmer ----------
STEM_EXC_AWAL = {'setuju', 'kesempatan', 'perusahaan', 'penyandang', 'menangis', 'bali', 'pisan',
                 'tuli', 'bisu', 'tunarungu', 'disabilitas', 'difabel', 'diskriminasi'}
STEM_EXC_BARU = STEM_EXC_AWAL | {
    'kekurangan', 'kelebihan', 'keterbatasan', 'pemerintah', 'lapangan',
    'bekasi', 'pakuan', 'pengunjung', 'menyinari', 'mengasih',
    # REVISI: Sastrawi salah memotong
    'semoga', 'meninggal', 'idaman', 'sekedar', 'sekadar', 'mentri', 'kusus', 'berbasis',
    'eksis', 'rasis', 'gelis', 'keterbasan', 'didikan', 'berasa', 'perasaan',
    'pembelinya', 'berkenalan', 'lainya', 'anaku', 'memilah', 'perawakan',
    'terkesan', 'mengakui', 'ajakan'
}
_stemmer = StemmerFactory().create_stemmer()
_stem_cache = {}

print(f"Kamus: {len(NORM):,} entri | Stopword awal: {len(STOP_AWAL)} -> baru: {len(STOP_BARU)} "
      f"({len(KEEP_WORDS)} kata dipertahaman) | Kata dasar: {len(ROOT_WORDS):,}")

Kamus: 4,387 entri | Stopword awal: 766 -> baru: 713 (71 kata dipertahaman) | Kata dasar: 29,932


In [56]:
import unicodedata
import re
import emoji

# Huruf 'small caps' (mis. "ᴘᴀs ɴɢᴏᴍᴏɴɢ") -> huruf biasa. Tanpa ini komentar jadi kosong setelah cleaning.
_SMALLCAP = str.maketrans('ᴀʙᴄᴅᴇꜰɢʜɪᴊᴋʟᴍɴᴏᴘǫʀꜱᴛᴜᴠᴡʏᴢ', 'abcdefghijklmnopqrstuvwyz')

def normalisasi_unicode(text):
    text = text.translate(_SMALLCAP)
    return unicodedata.normalize('NFKC', text)   # juga mengubah huruf tebal/miring/fullwidth ke huruf biasa

# 1) Case folding
def case_folding(text):
    return normalisasi_unicode(text).lower()

# 2) Cleaning
def cleaning(text):
    text = re.sub(r'[\u200b-\u200d\ufeff]', '', text)                  # karakter zero-width
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)                  # URL
    text = re.sub(r'@(?:[\w.\-]*?\d{4}|[\w.\-]+)', ' ', text)           # mention (handle YouTube: nama + 4 digit)
    text = emoji.replace_emoji(text, replace=' ')                       # emoji
    text = re.sub(r'\btuna\s*[- ]?\s*(rungu|runggu|wicara|wucara|netra|grahita)\b', r'tuna\1', text)  # tuna rungu -> tunarungu
    text = re.sub(r'[^a-z\s]', ' ', text)                               # angka & tanda baca
    text = re.sub(r'(.)\1{2,}', r'\1', text)                            # "mantaaap" -> "mantap"
    return re.sub(r'\s+', ' ', text).strip()

# 3) Tokenization
def tokenization(text):
    return text.split()

# 4) Normalization
def _norm_token(w):
    for _ in range(2):                       # dua putaran: g -> enggak -> tidak
        v = NORM.get(w, w)
        if v == w:
            break
        w = v
    return w

def _fix_elongation(w):
    # "aamiin" -> "amin", "cantikk" -> "cantik" bila hasilnya kata dasar yang valid
    if re.search(r'(.)\1', w):
        c = re.sub(r'(.)\1+', r'\1', w)
        if c in ROOT_WORDS:
            return c
    return w

def normalization(tokens):
    out = []
    for w in tokens:
        v = _norm_token(w)
        if v == w and w not in ROOT_WORDS:
            f = _fix_elongation(w)
            if f != w:
                v = _norm_token(f)
        out.extend(_norm_token(p) for p in v.replace('-', ' ').split())   # frasa -> beberapa token
    return out

# 5) Stopword removal (juga membuang token 1 huruf)
def stopword_removal(tokens):
    stop = STOP_BARU | (ENTITAS_VIDEO if HAPUS_ENTITAS_VIDEO else set())
    return [w for w in tokens if w not in stop and len(w) > 1]

# 6) Stemming (cache + pengecualian, termasuk bentuk berakhiran -nya/-lah/-kah/-pun)
def stem_token(w, exc=None):
    exc = STEM_EXC_BARU if exc is None else exc
    if w in exc:
        return w
    base = re.sub(r'(nya|lah|kah|pun)$', '', w)
    if base in exc:
        return base
    if w not in _stem_cache:
        _stem_cache[w] = _stemmer.stem(w)
    return _stem_cache[w]

def stemming(tokens):
    return [w for w in (stem_token(t) for t in tokens) if len(w) > 1]

def proses_baru(text):
    return stemming(stopword_removal(normalization(tokenization(cleaning(case_folding(text))))))

In [57]:
contoh = [
    "Setuju banget!!     😊👍 cek https://youtu.be/xxxx ya",
    "Tidak semua orang tuli sulit bekerja",
    "Tuna Rungu bukan berarti gak bisa kerja, yg penting krn mereka bekerja keras!!! @budi",
    "Luar biasa❤",
    "@Pepen8298Cerdas mantaaap amiiin, makasih ya",
]
rows = []
for k in contoh:
    s1 = case_folding(k); s2 = cleaning(s1); s3 = tokenization(s2); s4 = normalization(s3)
    s5 = stopword_removal(s4); s6 = stemming(s5)
    rows.append({'asli': k, '1 case folding': s1, '2 cleaning': s2, '3 tokenization': s3,
                 '4 normalization': s4, '5 stopword': s5, '6 stemming': s6})
pd.DataFrame(rows).T

,0,1,2,3,4
asli,Setuju banget!! 😊👍 cek https://youtu.be/xxxx ya,Tidak semua orang tuli sulit bekerja,"Tuna Rungu bukan berarti gak bisa kerja, yg penting krn mereka bekerja keras!!! @budi",Luar biasa❤,"@Pepen8298Cerdas mantaaap amiiin, makasih ya"
1 case folding,setuju banget!! 😊👍 cek https://youtu.be/xxxx ya,tidak semua orang tuli sulit bekerja,"tuna rungu bukan berarti gak bisa kerja, yg penting krn mereka bekerja keras!!! @budi",luar biasa❤,"@pepen8298cerdas mantaaap amiiin, makasih ya"
2 cleaning,setuju banget cek ya,tidak semua orang tuli sulit bekerja,tunarungu bukan berarti gak bisa kerja yg penting krn mereka bekerja keras,luar biasa,cerdas mantap amin makasih ya
3 tokenization,"[setuju, banget, cek, ya]","[tidak, semua, orang, tuli, sulit, bekerja]","[tunarungu, bukan, berarti, gak, bisa, kerja, yg, penting, krn, mereka, bekerja, keras]","[luar, biasa]","[cerdas, mantap, amin, makasih, ya]"
4 normalization,"[setuju, banget, cek, ya]","[tidak, semua, orang, tuli, sulit, bekerja]","[tunarungu, bukan, berarti, tidak, bisa, kerja, yang, penting, karena, mereka, bekerja, keras]","[luar, biasa]","[cerdas, mantap, amin, terima, kasih, ya]"
5 stopword,"[setuju, banget, cek]","[tidak, orang, tuli, sulit, bekerja]","[tunarungu, bukan, tidak, bisa, kerja, penting, mereka, bekerja, keras]","[luar, biasa]","[cerdas, mantap, amin, terima, kasih]"
6 stemming,"[setuju, banget, cek]","[tidak, orang, tuli, sulit, kerja]","[tunarungu, bukan, tidak, bisa, kerja, penting, mereka, kerja, keras]","[luar, biasa]","[cerdas, mantap, amin, terima, kasih]"


### Penerapan dataset

In [58]:
t0 = time.time()
df_cleaned['text_casefold'] = df_cleaned['text'].map(case_folding)
df_cleaned['text_clean']    = df_cleaned['text_casefold'].map(cleaning)
df_cleaned['tokens']        = df_cleaned['text_clean'].map(tokenization)
df_cleaned['tokens_norm']   = df_cleaned['tokens'].map(normalization)
df_cleaned['tokens_stop']   = df_cleaned['tokens_norm'].map(stopword_removal)
df_cleaned['tokens_stem']   = df_cleaned['tokens_stop'].progress_map(stemming)
df_cleaned['clean_text']    = df_cleaned['tokens_stem'].map(' '.join)
print(f"Selesai dalam {time.time() - t0:.1f} detik")
df_cleaned[['text', 'clean_text', LABEL_COL]].sample(10, random_state=RANDOM_STATE)

  0%|          | 0/13613 [00:00<?, ?it/s]

Selesai dalam 11.2 detik


,text,clean_text,label_annotator
6844,Semoga Matahari terus berjaya,semoga matahari jaya,inklusi
12808,"Mereka calon istri idaman,dibalik kekurangannya mempunyai semangat pekerja keras,,",mereka calon istri idaman balik kekurangan semangat kerja keras,inklusi
8604,"Jadii film drachin yg bos minimarket yang hebat ituuu ada versi real live nyaa,,,😌 difabel direkrut lgsg rame pembel...",film drachin bos minimarket hebat versi real live difabel rekrut langsung ramai beli,eksklusi
99,mereka berisik di keheningan,mereka berisik hening,inklusi
3780,klu gubernur syg rakyt ya kya gini.,gubernur sayang rakyat kayak,inklusi
3024,"MATAHARI KALIAN THE BEST , MEMPEKERJAKAN ORANG DIFABEL DENGAN BAIK , GOOD JOBB",matahari kalian best kerja orang difabel baik good jobb,inklusi
10310,Matahari Department store 👍,matahari department store,inklusi
7313,"Alhamdulillah,,,akhirnya di undang pula KDM,, 👍👍👍👍",alhamdulillah undang kdm,eksklusi
6479,"ASSALAMUALAIKUM PA KDM,SEMOGA SEHAT ,KALO DI PANJANGKAN UMUR INSYA ALLAH INGIN KETEMU PENGEN SILATUROHIM😊❤",assalamualaikum kdm semoga sehat panjang umur insya allah ingin ketemu pengin silaturohim,inklusi
1512,"PAK DEDY COBA CEK"" ADA NENEK NENEK MENINGGAL DI PUKULI GARA GARA MENCURI 2 BIJI LABU BUAT MAKAN""😭DI CIANJUR",dedy coba cek nenek nenek meninggal pukul gara gara curi biji labu makan cianjur,eksklusi


In [59]:
kosong = df_cleaned['clean_text'].str.strip() == ''
print(f"Komentar kosong setelah preprocessing: {kosong.sum()} baris "
      f"(label: {df_cleaned.loc[kosong, LABEL_COL].value_counts().to_dict()})")
print(df_cleaned.loc[kosong, ['text', LABEL_COL]].head(10))

df_final = df_cleaned.loc[~kosong].reset_index(drop=True)
print("\nJumlah data final:", len(df_final))
print(df_final[LABEL_COL].value_counts())

Komentar kosong setelah preprocessing: 48 baris (label: {'eksklusi': 24, 'inklusi': 24})
                                                            text  \
202                                                     Kedua🎉🎉🎉   
203                                                      pertama   
428   https://youtube.com/shorts/K9Elq0Hwz0Y?si=zkqOMuvOcnlTlGAs   
670                                           @spica30-b2w waduh   
783                                           @ainawaff235 🥹❤️‍🩹   
1520                                                   Wkwkwk😂😂😂   
2933                                                      Wkwkwk   
3256                                            Bapak aiiing❤❤❤❤   
3498                                                        Aing   
3516                                 ​@AdeSunarya-w4gterimakasih   

     label_annotator  
202         eksklusi  
203         eksklusi  
428         eksklusi  
670         eksklusi  
783          inklusi  
1520        eksklusi  
2

### HASIL PREPRO

In [60]:
KAMUS_FORMAL = set(kamus['formal'].dropna().astype(str)) | set(NORM.values())
tok = Counter(w for ts in df_cleaned['tokens_norm'] for w in ts)
oov = [(w, c) for w, c in tok.items()
       if w not in ROOT_WORDS and w not in KAMUS_FORMAL and w not in STOP_BARU and len(w) > 2]
oov.sort(key=lambda x: -x[1])
print(f"Token unik belum dikenal: {len(oov):,} dari {len(tok):,}")
pd.DataFrame(oov[:100], columns=['token', 'frekuensi'])

Token unik belum dikenal: 8,840 dari 12,947


,token,frekuensi
0,kdm,1397
1,hasni,525
2,store,497
3,hasna,490
4,dedi,438
...,...,...
95,ibunya,30
96,berjaya,30
97,bali,29
98,pakuan,29


In [61]:
sebelum = Counter(w for ts in df_cleaned['tokens_stop'] for w in ts)
rows = [(w, stem_token(w), c) for w, c in sebelum.items() if stem_token(w) != w]
audit = pd.DataFrame(rows, columns=['kata', 'hasil_stem', 'frekuensi']).sort_values('frekuensi', ascending=False)
print(f"Kata unik berubah: {len(audit):,} dari {len(sebelum):,}")
display(audit.head(100))
print("\nDicurigai terpotong terlalu pendek (kata >= 6 huruf menjadi <= 3 huruf):")
display(audit[(audit['kata'].str.len() >= 6) & (audit['hasil_stem'].str.len() <= 3)].head(40))

Kata unik berubah: 3,388 dari 12,458


,kata,hasil_stem,frekuensi
117,menerima,terima,294
7,mengomong,omong,228
145,bekerja,kerja,226
36,terharu,haru,190
451,pemimpin,pimpin,164
...,...,...,...
108,berusaha,usaha,22
434,menikahi,nikah,22
201,mudahkan,mudah,22
196,suaminya,suami,22



Dicurigai terpotong terlalu pendek (kata >= 6 huruf menjadi <= 3 huruf):


,kata,hasil_stem,frekuensi
17,berdua,dua,126
243,apapun,apa,41
239,tuanya,tua,32
594,ibunya,ibu,30
794,duanya,dua,20
429,doakan,doa,16
156,berhak,hak,13
1741,kemauan,mau,11
518,berdoa,doa,11
515,mendoakan,doa,7


In [62]:
dup_mask = df_final.duplicated('clean_text', keep=False)
n_label = df_final[dup_mask].groupby('clean_text')[LABEL_COL].nunique()
konflik = n_label[n_label > 1].index
print(f"Baris dengan clean_text kembar : {dup_mask.sum():,}")
print(f"clean_text kembar beda label   : {len(konflik):,}")
display(df_final[df_final['clean_text'].isin(konflik)][['text', 'clean_text', LABEL_COL]]
        .sort_values('clean_text').head(30))

HAPUS_DUPLIKAT_CLEAN = True   # duplikat = clean_text sama DAN label sama
if HAPUS_DUPLIKAT_CLEAN:
    sebelum_n = len(df_final)
    df_final = df_final.drop_duplicates(subset=['clean_text', LABEL_COL]).reset_index(drop=True)
    print(f"\nDuplikat dihapus: {sebelum_n - len(df_final):,} -> data final {len(df_final):,}")
print(df_final[LABEL_COL].value_counts())

Baris dengan clean_text kembar : 1,838
clean_text kembar beda label   : 198


,text,clean_text,label_annotator
11037,Saya minta alamat nya,alamat,eksklusi
11043,Saya minta alamat ny dong,alamat,eksklusi
11186,Minta alamatnya,alamat,inklusi
11145,BG minta alamat nyah,alamat,inklusi
11129,ne alamat nya dimana,alamat mana,eksklusi
10809,"Bang ,alamatnya di mana \n\n?",alamat mana,eksklusi
11067,Alamat nya di mana,alamat mana,eksklusi
10565,Alamatnya dimana y,alamat mana,eksklusi
10439,Alamatnya di mna,alamat mana,eksklusi
11194,Alamat nya di mana bang,alamat mana,inklusi



Duplikat dihapus: 1,228 -> data final 12,337
label_annotator
inklusi     8528
eksklusi    3809
Name: count, dtype: int64


In [63]:
jml = df_final['clean_text'].str.split().str.len()
print(f"Data final        : {len(df_final):,}")
print(f"Rata-rata token   : {jml.mean():.1f} | median {jml.median():.0f} | maks {jml.max()}")
print(f"Komentar 1-2 token: {(jml <= 2).sum():,} ({(jml <= 2).mean():.1%})")
print(f"Ukuran kosakata   : {len(set(w for t in df_final['clean_text'] for w in t.split())):,}")
df_final[['text', 'clean_text', LABEL_COL]].to_csv('/content/drive/MyDrive/data_preprocessed.csv', index=False)
print("Tersimpan: data_preprocessed.csv")

Data final        : 12,337
Rata-rata token   : 7.7 | median 6 | maks 332
Komentar 1-2 token: 1,714 (13.9%)
Ukuran kosakata   : 9,686
Tersimpan: data_preprocessed.csv
